In [0]:
-- =====================================================================
-- CineClub 2027 Scorecard — P12M baselines (2025-10-01 to 2026-09-30)
-- Members = current active (latest snapshot, renewal_status LIKE 'Active%')
-- Member activity = CineClubFLAG = 'Yes' lines
-- =====================================================================


-- 1. AVG VISITS PER MEMBER (P12M) -------------------------------------
WITH active_members AS (
    SELECT DISTINCT cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                              FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
      AND renewal_status LIKE 'Active%'
      AND cde_id > 0
),
cc_visits AS (
    SELECT t.CDE_ID, t.VisitDateId, t.LocationId
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS t
    INNER JOIN active_members AS m ON m.cde_id = t.CDE_ID
    WHERE t.VisitDateId BETWEEN 20251001 AND 20260930
      AND t.CineClubFLAG = 'Yes'
    GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId
    HAVING SUM(t.Quantity) > 0
)
SELECT
    (SELECT COUNT(*) FROM active_members)                      AS active_members,
    COUNT(*)                                                   AS cc_visits,
    ROUND(COUNT(*) / (SELECT COUNT(*) FROM active_members), 2) AS avg_visits_per_member
FROM cc_visits;


-- 2. ANNUAL RENEWAL RATE (P12M) ---------------------------------------
-- Events-based (snapshot renewal_datetime is unreliable before ~Mar 2026)
-- Due     = an annual1 term that ended (start + 1 year) in P12M
-- Renewed = a later annual1 payment whose term starts within +/-5 days of that end date
-- Result on 2026-10-28: 41,694 / 55,079 = 75.7%
WITH annual_terms AS (
    SELECT DISTINCT cc_user_profile_id,
           to_date(current_term_start_datetime) AS term_start
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
      AND current_term_start_datetime IS NOT NULL
),
due AS (
    SELECT cc_user_profile_id, add_months(term_start, 12) AS renewal_date
    FROM annual_terms
    WHERE add_months(term_start, 12) BETWEEN DATE'2025-10-01' AND DATE'2026-09-30'
)
SELECT
    date_format(d.renewal_date, 'yyyyMM')                                         AS yyyymm,
    COUNT(DISTINCT d.cc_user_profile_id)                                          AS due,
    COUNT(DISTINCT CASE WHEN n.cc_user_profile_id IS NOT NULL THEN d.cc_user_profile_id END) AS renewed,
    ROUND(COUNT(DISTINCT CASE WHEN n.cc_user_profile_id IS NOT NULL THEN d.cc_user_profile_id END)
          / COUNT(DISTINCT d.cc_user_profile_id) * 100, 1)                        AS renewal_rate_pct
FROM due AS d
LEFT JOIN annual_terms AS n
    ON  n.cc_user_profile_id = d.cc_user_profile_id
    AND n.term_start BETWEEN date_sub(d.renewal_date, 5) AND date_add(d.renewal_date, 5)
GROUP BY ROLLUP(date_format(d.renewal_date, 'yyyyMM'))
ORDER BY yyyymm;


-- 3. AVG CONCESSION SPEND PER VISIT: CineClub vs Scene+ non-CineClub (P12M)
-- CineClub visit = current active member, CineClubFLAG = 'Yes'
-- Scene+ non-CC visit = SCENEFLAG = 'Yes' and CineClubFLAG IS DISTINCT FROM 'Yes'
WITH active_members AS (
    SELECT DISTINCT cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                              FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
      AND renewal_status LIKE 'Active%'
      AND cde_id > 0
),
visits AS (
    SELECT
        CASE WHEN t.CineClubFLAG = 'Yes' THEN 'CineClub' ELSE 'Scene+ non-CineClub' END AS cohort,
        t.CDE_ID, t.VisitDateId, t.LocationId,
        SUM(CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END) AS concession_revenue
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS t
    INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location AS l ON l.LocationID = t.LocationId
    LEFT JOIN active_members AS m ON m.cde_id = t.CDE_ID
    WHERE t.VisitDateId BETWEEN 20251001 AND 20260930
      AND l.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.RecRoomFLAG = 'No' AND t.STOREFLAG = 'No'
      AND t.CDE_ID > 0
      AND (   (t.CineClubFLAG = 'Yes' AND m.cde_id IS NOT NULL)
           OR (t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes'))
    GROUP BY 1, t.CDE_ID, t.VisitDateId, t.LocationId
    HAVING SUM(t.Quantity) > 0
)
SELECT
    cohort,
    COUNT(*)                                     AS visits,
    ROUND(SUM(concession_revenue), 0)            AS concession_revenue,
    ROUND(SUM(concession_revenue) / COUNT(*), 2) AS concession_spend_per_visit
FROM visits
GROUP BY cohort;


-- =====================================================================
-- VALIDATION
-- =====================================================================

-- V1. Active member count + snapshot date used (should match the dashboard balance)
SELECT MAX(snapshot_date_id) AS snapshot_used, COUNT(DISTINCT cde_id) AS active_members
FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                          FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
  AND renewal_status LIKE 'Active%' AND cde_id > 0;

-- V2. Renewal rate by month: look for a month that breaks the pattern
SELECT substr(snapshot_date_id, 1, 6) AS yyyymm, COUNT(DISTINCT cc_user_profile_id) AS due
FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
WHERE snapshot_date_id IN ('20251001','20251101','20251201','20260101','20260201','20260301',
                           '20260401','20260501','20260601','20260701','20260801','20260901')
  AND plan_id = 'annual1' AND renewal_status LIKE 'Active%'
  AND date_format(renewal_datetime, 'yyyyMM') = substr(snapshot_date_id, 1, 6)
GROUP BY 1 ORDER BY 1;

-- Avg visits per visiting guest, P12M: CineClub vs Scene+ non-CineClub
-- Same visit + cohort logic as query 3
WITH active_members AS (
    SELECT DISTINCT cde_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                              FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
      AND renewal_status LIKE 'Active%'
      AND cde_id > 0
),
visits AS (
    SELECT
        CASE WHEN t.CineClubFLAG = 'Yes' THEN 'CineClub' ELSE 'Scene+ non-CineClub' END AS cohort,
        t.CDE_ID, t.VisitDateId, t.LocationId
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction AS t
    INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location AS l ON l.LocationID = t.LocationId
    LEFT JOIN active_members AS m ON m.cde_id = t.CDE_ID
    WHERE t.VisitDateId BETWEEN 20251001 AND 20260930
      AND l.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.RecRoomFLAG = 'No' AND t.STOREFLAG = 'No'
      AND t.CDE_ID > 0
      AND (   (t.CineClubFLAG = 'Yes' AND m.cde_id IS NOT NULL)
           OR (t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes'))
    GROUP BY 1, t.CDE_ID, t.VisitDateId, t.LocationId
    HAVING SUM(t.Quantity) > 0
)
SELECT
    cohort,
    COUNT(DISTINCT CDE_ID)                              AS guests,
    COUNT(*)                                            AS visits,
    ROUND(COUNT(*) / COUNT(DISTINCT CDE_ID), 2)         AS avg_visits_per_guest
FROM visits
GROUP BY cohort;